In [3]:
import torch
import importlib
import attention
import load_data
import torch.nn as nn

# Reload both modules
importlib.reload(attention)
importlib.reload(load_data)

# Re-import after reload
from attention import DualAttentionSeq2Seq
from load_data import get_data_loaders

# Load data (Latin → Devanagari)
train_loader, dev_loader, test_loader, char_to_idx_latin, char_to_idx_devanagari = get_data_loaders()

# Vocabulary info
input_vocab_size = len(char_to_idx_latin)
output_vocab_size = len(char_to_idx_devanagari)
idx_to_input = {v: k for k, v in char_to_idx_latin.items()}
idx_to_output = {v: k for k, v in char_to_idx_devanagari.items()}
# Index maps
idx_to_latin = {v: k for k, v in char_to_idx_latin.items()}
idx_to_devanagari = {v: k for k, v in char_to_idx_devanagari.items()}
# Device
device = torch.device("cuda")

# Model
model = DualAttentionSeq2Seq(
    input_vocab_size=input_vocab_size,
    output_vocab_size=output_vocab_size,
    embedding_dim=64,
    hidden_dim=64,
    num_layers=2,
    num_heads=2,
    dropout=0.2,
    device=device
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters())

# Training loop
best_val_loss = float('inf')
for epoch in range(2):
    model.train()
    train_loss = 0
    
    for src, trg in train_loader:
        src, trg = src.to(device), trg.to(device)

        output = model(src, trg[:, :-1])  # teacher forcing
        loss = criterion(
            output.reshape(-1, output_vocab_size),
            trg[:, 1:].reshape(-1)
        )
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item()
    
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for src, trg in dev_loader:
            src, trg = src.to(device), trg.to(device)
            output = model(src, trg[:, :-1])
            val_loss += criterion(
                output.reshape(-1, output_vocab_size),
                trg[:, 1:].reshape(-1)
            ).item()
    
    val_loss /= len(dev_loader)
    print(f'Epoch {epoch+1}: Train Loss = {train_loss/len(train_loader):.4f}, Val Loss = {val_loss:.4f}')
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_attention_model.pt')
        print("Saved new best model")

# Load best model
model.load_state_dict(torch.load('best_attention_model.pt'))

# Helper: Decode sequence
def decode_sequence(indices, idx_to_char):
    return ''.join([idx_to_char.get(idx, '') for idx in indices if idx not in {0, 1, 2}])

# Testing loop
model.eval()
test_loss = 0
correct = 0
total = 0
examples = []

with torch.no_grad():
    for src, trg in test_loader:
        src, trg = src.to(device), trg.to(device)
        output = model(src, trg[:, :-1])

        # Loss
        test_loss += criterion(
            output.reshape(-1, output_vocab_size),
            trg[:, 1:].reshape(-1)
        ).item()
        
        # Accuracy (with safe min length)
        _, predicted = output.max(2)
        min_len = min(predicted.size(1), trg[:, 1:].size(1))
        correct += (predicted[:, :min_len] == trg[:, 1:][:, :min_len]).sum().item()
        total += (trg[:, 1:][:, :min_len] != 0).sum().item()
        
        # Save sample predictions
        for i in range(min(3, src.size(0))):
            input_seq = decode_sequence(src[i].cpu().numpy(), idx_to_latin)
            pred_seq = decode_sequence(predicted[i].cpu().numpy(), idx_to_devanagari)
            true_seq = decode_sequence(trg[i].cpu().numpy(), idx_to_devanagari)
            examples.append((input_seq, pred_seq, true_seq))

# Report
test_loss /= len(test_loader)
accuracy = correct / total
print(f'\nTest Loss: {test_loss:.4f}, Accuracy: {accuracy:.2%}')

print("\nExample Predictions:")
for i, (input_seq, pred_seq, true_seq) in enumerate(examples[:10]):
    status = "✓" if pred_seq == true_seq else "✗"
    print(f"{i+1}. Input: {input_seq}")
    print(f"   Pred: {pred_seq}")
    print(f"   True: {true_seq} ({status})\n")

# Save all test predictions to a file
with open("predictions_attention.tsv", "w", encoding="utf-8") as f:
    f.write("latin\tpredicted\tground_truth\n")
    for input_seq, pred_seq, true_seq in examples:
        f.write(f"{input_seq}\t{pred_seq}\t{true_seq}\n")
print("\nSaved all predictions to predictions_attention.tsv")


Epoch 1: Train Loss = 1.1383, Val Loss = 0.0643
Saved new best model
Epoch 2: Train Loss = 0.2573, Val Loss = 0.0475
Saved new best model

Test Loss: 0.0422, Accuracy: 99.15%

Example Predictions:
1. Input: ank
   Pred: अंक
   True: अंक (✓)

2. Input: anka
   Pred: अंक
   True: अंक (✓)

3. Input: ankit
   Pred: अंकित
   True: अंकित (✓)

4. Input: augustine
   Pred: अगस्टाइनन
   True: अगस्टाइन (✗)

5. Input: augustustine
   Pred: अगस्टाइनन
   True: अगस्टाइन (✗)

6. Input: agsta
   Pred: अगस्ता
   True: अगस्ता (✓)

7. Input: atthas
   Pred: अट्टहास
   True: अट्टहास (✓)

8. Input: addon
   Pred: अड्डों
   True: अड्डों (✓)

9. Input: athak
   Pred: अथक
   True: अथक (✓)

10. Input: anik
   Pred: अनिकअ
   True: अनिक (✗)


Saved all predictions to predictions_attention.tsv
